# Análisis interno de ESM-2: Tokens, Embeddings, Atención y Representaciones

**Universidad de Pamplona — Ciencia de Datos 2026-1**
**Diego Alejandro Contreras Contreras CC: 1091354438**

Este notebook cubre todas las actividades obligatorias del Trabajo de Segundo Corte.

**Modelo usado:** `facebook/esm2_t12_35M_UR50D`
- 6 capas transformer
- 8 millones de parámetros
- 20 cabezas de atención por capa
- Diseñado para secuencias de aminoácidos

---
### Tabla de contenidos
1. Instalación y configuración del entorno
2. Carga del modelo y tokenizer
3. Actividad 2: Tokenización de las 3 secuencias
4. Actividad 2: Inferencia y extracción de hidden states
5. Actividad 3: Inspección del código fuente
6. Actividad 4: Visualización de embeddings y similitud coseno
7. Actividad 5: Matrices de atención
8. Actividad 6: Masked Language Modeling
9. Actividad 7: Resumen de usos reales y limitaciones

---
## Celda 1 — Instalación del entorno
Instalamos las librerías necesarias. En Google Colab `torch` ya viene preinstalado,
pero instalamos `transformers` y `scikit-learn` explícitamente para garantizar compatibilidad.

In [ ]:
# ============================================================
# CELDA 1: INSTALACIÓN
# ============================================================
# Instalamos / actualizamos las librerías que vamos a usar.
# El flag -q suprime la salida larga de pip para mantener el notebook limpio.

!pip install -q transformers>=4.40.0 scikit-learn matplotlib seaborn

print("✅ Librerías instaladas correctamente")

---
## Celda 2 — Importaciones
Importamos todo lo que necesitamos de una vez.
Así cualquier error de importación aparece al inicio y no a mitad del notebook.

In [ ]:
# ============================================================
# CELDA 2: IMPORTACIONES
# ============================================================

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from transformers import AutoTokenizer, EsmModel, EsmForMaskedLM
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

# Verificamos si hay GPU disponible en Colab
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Dispositivo en uso: {device}")
print(f"🔧  Versión de PyTorch: {torch.__version__}")

import transformers
print(f"🤗  Versión de Transformers: {transformers.__version__}")

---
## Celda 3 — Carga del modelo y tokenizer
Cargamos el modelo preentrenado `facebook/esm2_t6_8M_UR50D` desde Hugging Face.
La primera vez descarga los pesos (~30 MB). Las siguientes veces usa caché.

In [ ]:
# ============================================================
# CELDA 3: CARGA DEL MODELO
# ============================================================

MODEL_NAME = "facebook/esm2_t12_35M_UR50D"

print(f" Cargando tokenizer de {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f" Cargando modelo EsmModel (para embeddings y atenciones)...")
model = EsmModel.from_pretrained(MODEL_NAME)
model = model.to(device)
model.eval()  # Modo evaluación: desactiva dropout, no calcula gradientes

# Inspeccionamos la arquitectura del modelo
print("\n" + "="*60)
print("📐 ARQUITECTURA DEL MODELO")
print("="*60)
print(f"  Número de capas (encoder layers): {model.config.num_hidden_layers}")
print(f"  Número de cabezas de atención:    {model.config.num_attention_heads}")
print(f"  Tamaño del embedding (hidden size):{model.config.hidden_size}")
print(f"  Tamaño del vocabulario:           {model.config.vocab_size}")
print(f"  Total de parámetros: {sum(p.numel() for p in model.parameters()):,}")

---
## Celda 4 — Definición de las 3 secuencias (Actividad 2)
Usamos las secuencias del Anexo B del documento guía:
- **Original**: proteína de referencia
- **Mutada**: cambio puntual Y→F en posición 5 (tirosina por fenilalanina)
- **Alterada**: inversión parcial de la secuencia, usada como control

In [ ]:
# ============================================================
# CELDA 4: DEFINICIÓN DE SECUENCIAS
# ============================================================

sequences = {
    "Original": "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG",

    "Mutada": "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGA",
    # G → A en la última posición (posición 76), mutación G76A real y documentada

    "Alterada": "GGRLRLVLHLTSEKQINYDSLTRGDELQKGAFIRLQQPDPPIGEQKIQDKAKKVNEPSDTIENVTLTGKTFVIQM",
    # Inversión parcial de la secuencia original, como control
}
print("SECUENCIAS DE TRABAJO")
print("-"*60)
for nombre, seq in sequences.items():
    print(f"  {nombre:10s} ({len(seq)} aa): {seq}")

# Marcamos la diferencia entre original y mutada
orig = sequences["Original"]
mut  = sequences["Mutada"]
print("\n Diferencia original vs mutada:")
for i, (a, b) in enumerate(zip(orig, mut)):
    if a != b:
        print(f"   Posición {i+1}: '{a}' → '{b}' (cambio puntual)")

---
## Celda 5 — Tokenización detallada (Actividad 2)
El tokenizer convierte cada aminoácido en un ID numérico que el modelo puede procesar.
ESM-2 agrega dos tokens especiales: `<cls>` al inicio y `<eos>` al final.

In [ ]:
# ============================================================
# CELDA 5: TOKENIZACIÓN DETALLADA
# ============================================================

print("ANÁLISIS DE TOKENIZACIÓN POR SECUENCIA")
print("="*60)

tokenized_data = {}  # Guardamos los inputs para usarlos luego

for nombre, seq in sequences.items():
    inputs = tokenizer(seq, return_tensors="pt")
    input_ids   = inputs["input_ids"][0]  # IDs numéricos de cada token
    tokens      = tokenizer.convert_ids_to_tokens(input_ids)  # Texto de cada token

    tokenized_data[nombre] = inputs  # Guardamos para inferencia

    print(f"\n {nombre} ({len(seq)} aminoácidos)")
    print(f"   Tokens totales (con especiales): {len(input_ids)}")
    print(f"   Tokens: {tokens}")
    print(f"   IDs:    {input_ids.tolist()}")
    print(f"   Shape del tensor input_ids: {inputs['input_ids'].shape}")
    print(f"   (batch_size=1, sequence_length={inputs['input_ids'].shape[1]})")

---
## Celda 6 — Inferencia: extracción de hidden states y atenciones (Actividad 2)
Pasamos las 3 secuencias por el modelo con `output_hidden_states=True` y `output_attentions=True`.

Obtenemos:
- `last_hidden_state`: representación contextualizada de la última capa
- `hidden_states`: representaciones de TODAS las capas (incluida la capa 0 = embeddings iniciales)
- `attentions`: matrices de atención de cada capa

In [ ]:
# ============================================================
# CELDA 6: INFERENCIA CON EXTRACCIÓN DE TODOS LOS OUTPUTS
# ============================================================
from transformers import AutoConfig

config = AutoConfig.from_pretrained(MODEL_NAME)
config.output_attentions = True

model = EsmModel.from_pretrained(MODEL_NAME, config=config)
model = model.to(device)
model.eval()

outputs_dict = {}  # Guardamos los outputs de cada secuencia

print(" INFERENCIA Y EXTRACCIÓN DE REPRESENTACIONES")
print("="*60)

for nombre, seq in sequences.items():
    inputs = tokenized_data[nombre]
    inputs_on_device = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():  # No calculamos gradientes (solo inferencia)
        outputs = model(
            **inputs_on_device,
            output_hidden_states=True,   # Queremos TODAS las capas
            output_attentions=True       # Queremos las matrices de atención
        )

    outputs_dict[nombre] = outputs

    print(f"\n{nombre}")
    print(f"   last_hidden_state shape: {outputs.last_hidden_state.shape}")
    print(f"   → (batch=1, tokens={outputs.last_hidden_state.shape[1]}, hidden_size={outputs.last_hidden_state.shape[2]})")
    print(f"   Número de hidden_states: {len(outputs.hidden_states)} (capa 0 + 6 capas transformer)")

    if outputs.attentions is not None:
        print(f"   Número de tensores de atención: {len(outputs.attentions)}")
        print(f"   Shape de atención [capa 0]: {outputs.attentions[0].shape}")
        print(f"   → (batch=1, heads={outputs.attentions[0].shape[1]}, seq_len, seq_len)")
    else:
        print("    outputs.attentions es None en esta configuración")

print("\n Inferencia completa para las 3 secuencias")

---
## Celda 7 — Inspección del código fuente (Actividad 3)
Usamos Python para inspeccionar directamente las clases del modelo: vemos sus atributos,
sus módulos internos y cómo están conectados.

In [ ]:
# ============================================================
# CELDA 7A: INSPECCIÓN DEL TOKENIZER
# ============================================================

print(" INSPECCIÓN DEL TOKENIZER")
print("="*60)
print(f"  Clase: {type(tokenizer).__name__}")
print(f"  Tamaño del vocabulario: {tokenizer.vocab_size}")
print(f"  Token de máscara: '{tokenizer.mask_token}' → ID={tokenizer.mask_token_id}")
print(f"  Token CLS:        '{tokenizer.cls_token}'  → ID={tokenizer.cls_token_id}")
print(f"  Token EOS:        '{tokenizer.eos_token}'  → ID={tokenizer.eos_token_id}")
print(f"  Token PAD:        '{tokenizer.pad_token}'  → ID={tokenizer.pad_token_id}")

# Mostramos parte del vocabulario (los aminoácidos)
print("\n  Fragmento del vocabulario (aminoácidos):")
vocab = tokenizer.get_vocab()
aa_tokens = {k: v for k, v in vocab.items() if len(k)==1 and k.isupper()}
print(f"  {aa_tokens}")

In [ ]:
# ============================================================
# CELDA 7B: INSPECCIÓN DE LOS MÓDULOS DEL MODELO
# ============================================================

print(" MÓDULOS PRINCIPALES DEL MODELO ESM-2")
print("="*60)

# Listamos los subcomponentes de primer nivel
for name, module in model.named_children():
    print(f"\n  [{name}] → {type(module).__name__}")

    # Un nivel más de profundidad
    for subname, submodule in module.named_children():
        print(f"      [{subname}] → {type(submodule).__name__}")

In [ ]:
# ============================================================
# CELDA 7C: INSPECCIÓN DE LA CAPA DE ATENCIÓN
# ============================================================

print(" INSPECCIÓN DE LA CAPA DE SELF-ATTENTION (capa 0)")
print("="*60)

# Accedemos a la primera capa del encoder
attn_layer = model.encoder.layer[0].attention

print(f"  Clase de atención: {type(attn_layer).__name__}")
print(f"  Número de cabezas: {attn_layer.self.num_attention_heads}")
print(f"  Tamaño por cabeza: {attn_layer.self.attention_head_size}")
print(f"  Proyección Q: {attn_layer.self.query}")
print(f"  Proyección K: {attn_layer.self.key}")
print(f"  Proyección V: {attn_layer.self.value}")

print("\n🔍 INSPECCIÓN DEL MÓDULO DE EMBEDDINGS")
print("="*60)
embeddings_module = model.embeddings
print(f"  Clase: {type(embeddings_module).__name__}")
for name, mod in embeddings_module.named_children():
    print(f"  [{name}] → {type(mod).__name__}: {mod}")

In [ ]:
# ============================================================
# CELDA 7D: TABLA RESUMEN DE COMPONENTES
# ============================================================

print(" TABLA RESUMEN — COMPONENTES IDENTIFICADOS EN CÓDIGO")
print("="*80)

tabla = [
    ("Tokenizer",         "AutoTokenizer / EsmTokenizer",             "Convierte aminoácidos en IDs numéricos"),
    ("Embeddings",        "model.embeddings.word_embeddings",         "Convierte IDs en vectores de dim=480"),
    ("Pos. Embeddings",   "model.embeddings.position_embeddings",     "Añade información de posición en la secuencia"),
    ("Self-Attention",    "model.encoder.layer[i].attention.self",    "Calcula Q, K, V y pesos de atención"),
    ("Multi-head Att.",   "attn.self.query / .key / .value",         "Proyecciones paralelas para cada cabeza"),
    ("LayerNorm",         "model.encoder.layer[i].attention.output", "Normaliza y suma residual"),
    ("Feed-forward",      "model.encoder.layer[i].intermediate",     "Red MLP por posición (expande y contrae)"),
    ("MLM Head",          "EsmForMaskedLM: lm_head",                 "Produce logits para predecir tokens enmascarados"),
]

print(f"{'Componente':<20} {'Dónde en código':<40} {'Qué hace'}")
print("-"*80)
for comp, donde, que in tabla:
    print(f"{comp:<20} {donde:<40} {que}")

---
## Celda 8 — Embeddings globales y similitud coseno (Actividad 4)
Para representar una proteína completa como un solo vector, promediamos los
hidden states de todos los residuos (excluyendo tokens especiales CLS y EOS).

In [ ]:
# ============================================================
# CELDA 8: EMBEDDING GLOBAL POR PROTEÍNA + SIMILITUD COSENO
# ============================================================

print(" EMBEDDINGS GLOBALES (promedio de residuos, última capa)")
print("="*60)

global_embeddings = {}

for nombre, outputs in outputs_dict.items():
    # last_hidden_state shape: (1, seq_len, hidden_size)
    # Excluimos posición 0 (CLS) y la última (EOS), promediamos los residuos reales
    hidden = outputs.last_hidden_state[0]  # → (seq_len, hidden_size)
    seq_len = hidden.shape[0]

    # Promediamos solo los aminoácidos reales (sin CLS ni EOS)
    protein_embedding = hidden[1:seq_len-1].mean(dim=0)  # → (hidden_size,)
    global_embeddings[nombre] = protein_embedding.cpu().numpy()

    print(f"  {nombre}: embedding shape = {protein_embedding.shape}")
    print(f"           primeros 5 valores: {protein_embedding[:5].cpu().numpy().round(4)}")

# Calculamos similitud coseno entre los 3 embeddings globales
print("\n SIMILITUD COSENO ENTRE SECUENCIAS")
print("-"*40)
nombres = list(global_embeddings.keys())
matrix = np.array([global_embeddings[n] for n in nombres])
sim_matrix = cosine_similarity(matrix)

print(f"{'':12}", end="")
for n in nombres:
    print(f"{n:>12}", end="")
print()
for i, n in enumerate(nombres):
    print(f"{n:12}", end="")
    for j in range(len(nombres)):
        print(f"{sim_matrix[i,j]:>12.4f}", end="")
    print()

print("\n💡 Interpretación:")
print(f"  Original vs Mutada:   {sim_matrix[0,1]:.4f} (cambio puntual Y→F)")
print(f"  Original vs Alterada: {sim_matrix[0,2]:.4f} (inversión parcial severa)")

---
##  Celda 9 — Visualización PCA de embeddings por residuo (Actividad 4)

In [ ]:
# ============================================================
# CELDA 9: PCA DE EMBEDDINGS POR RESIDUO
# ============================================================

# Extraemos los embeddings por residuo de cada secuencia (sin CLS y EOS)
all_embeddings = []
all_labels     = []
all_residues   = []
colors_map = {"Original": "steelblue", "Mutada": "tomato", "Alterada": "seagreen"}

for nombre, outputs in outputs_dict.items():
    hidden = outputs.last_hidden_state[0].cpu().numpy()  # (seq_len, 320)
    seq = sequences[nombre]
    # Solo los residuos reales (excluimos CLS=pos0 y EOS=última)
    residue_embeddings = hidden[1:len(seq)+1]
    all_embeddings.append(residue_embeddings)
    all_labels.extend([nombre] * len(seq))
    all_residues.extend(list(seq))

X = np.vstack(all_embeddings)  # (total_residuos, 320)

# Reducimos a 2D con PCA
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X)

# Graficamos
fig, ax = plt.subplots(figsize=(10, 7))

start = 0
for nombre, outputs in outputs_dict.items():
    n = len(sequences[nombre])
    pts = X_2d[start:start+n]
    ax.scatter(pts[:, 0], pts[:, 1],
               label=nombre,
               color=colors_map[nombre],
               alpha=0.8, s=80, edgecolors='white', linewidth=0.5)
    # Anotamos el primer y último aminoácido
    seq = sequences[nombre]
    ax.annotate(f"{seq[0]}(0)",  pts[0],  fontsize=7, alpha=0.7)
    ax.annotate(f"{seq[-1]}(end)", pts[-1], fontsize=7, alpha=0.7)
    start += n

var_explained = pca.explained_variance_ratio_
ax.set_xlabel(f"PC1 ({var_explained[0]*100:.1f}% varianza explicada)", fontsize=11)
ax.set_ylabel(f"PC2 ({var_explained[1]*100:.1f}% varianza explicada)", fontsize=11)
ax.set_title("PCA de embeddings por residuo — última capa de ESM-2", fontsize=13)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("pca_embeddings.png", dpi=150, bbox_inches='tight')
plt.show()
print("\n💾 Figura guardada como pca_embeddings.png")

---
##  Celda 10 — Mapa de calor de similitud coseno entre secuencias (Actividad 4)

In [ ]:
# ============================================================
# CELDA 10: HEATMAP DE SIMILITUD COSENO
# ============================================================

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(sim_matrix, cmap="YlOrRd", vmin=0.5, vmax=1.0)
plt.colorbar(im, ax=ax, label="Similitud coseno")

ax.set_xticks(range(len(nombres)))
ax.set_yticks(range(len(nombres)))
ax.set_xticklabels(nombres, fontsize=11)
ax.set_yticklabels(nombres, fontsize=11)

for i in range(len(nombres)):
    for j in range(len(nombres)):
        ax.text(j, i, f"{sim_matrix[i,j]:.3f}",
                ha="center", va="center", fontsize=12,
                color="white" if sim_matrix[i,j] > 0.85 else "black")

ax.set_title("Similitud coseno entre embeddings globales", fontsize=12)
plt.tight_layout()
plt.savefig("cosine_similarity.png", dpi=150, bbox_inches='tight')
plt.show()
print("💾 Figura guardada como cosine_similarity.png")

---
##  Celda 11 — Matrices de atención (Actividad 5)
Visualizamos cómo cada token de la secuencia atiende a los demás tokens.
Cada fila = distribución de atención de un token hacia todos los otros.

In [ ]:
# ============================================================
# CELDA 11A: FUNCIÓN PARA GRAFICAR MATRICES DE ATENCIÓN
# ============================================================

def plot_attention(outputs, sequence_name, seq, layer_idx, head_idx, ax, title=None):
    """Grafica una matriz de atención para una capa y cabeza específica."""
    if outputs.attentions is None:
        ax.text(0.5, 0.5, "attentions=None\nVerificar configuración",
                ha='center', va='center', transform=ax.transAxes, fontsize=10)
        return

    # Obtenemos la matriz: shape (batch, heads, seq_len, seq_len)
    attn = outputs.attentions[layer_idx][0, head_idx].cpu().numpy()

    # Los tokens incluyen CLS y EOS, los recuperamos
    inputs = tokenized_data[sequence_name]
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    im = ax.imshow(attn, cmap="Blues", aspect="auto", vmin=0)
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=90, fontsize=7)
    ax.set_yticklabels(tokens, fontsize=7)
    t = title or f"{sequence_name} | Capa {layer_idx}, Cabeza {head_idx}"
    ax.set_title(t, fontsize=9)
    return im

In [ ]:
# ============================================================
# CELDA 11B: CAPA TEMPRANA vs CAPA PROFUNDA (secuencia original)
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

plot_attention(outputs_dict["Original"], "Original", sequences["Original"],
               layer_idx=0, head_idx=0, ax=axes[0],
               title="Original | Capa TEMPRANA (0), Cabeza 0")

plot_attention(outputs_dict["Original"], "Original", sequences["Original"],
               layer_idx=5, head_idx=0, ax=axes[1],
               title="Original | Capa PROFUNDA (5), Cabeza 0")

plt.suptitle("Comparación: capa temprana vs capa profunda", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("attn_early_vs_deep.png", dpi=150, bbox_inches='tight')
plt.show()
print(" Figura guardada como attn_early_vs_deep.png")

In [ ]:
# ============================================================
# CELDA 11C: DOS CABEZAS DISTINTAS EN LA MISMA CAPA
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

plot_attention(outputs_dict["Original"], "Original", sequences["Original"],
               layer_idx=3, head_idx=0, ax=axes[0],
               title="Original | Capa 3, Cabeza 0")

plot_attention(outputs_dict["Original"], "Original", sequences["Original"],
               layer_idx=3, head_idx=5, ax=axes[1],
               title="Original | Capa 3, Cabeza 5")

plt.suptitle("Misma capa (3), cabezas distintas → patrones de atención diferentes",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig("attn_two_heads.png", dpi=150, bbox_inches='tight')
plt.show()
print(" Figura guardada como attn_two_heads.png")

In [ ]:
# ============================================================
# CELDA 11D: ORIGINAL vs MUTADA (misma capa y cabeza)
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

plot_attention(outputs_dict["Original"], "Original", sequences["Original"],
               layer_idx=2, head_idx=1, ax=axes[0],
               title="ORIGINAL | Capa 2, Cabeza 1")

plot_attention(outputs_dict["Mutada"], "Mutada", sequences["Mutada"],
               layer_idx=2, head_idx=1, ax=axes[1],
               title="MUTADA (Y→F) | Capa 2, Cabeza 1")

plt.suptitle("Efecto de mutación puntual (Y→F) en matriz de atención",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig("attn_original_vs_mutada.png", dpi=150, bbox_inches='tight')
plt.show()
print(" Figura guardada como attn_original_vs_mutada.png")

---
##  Celda 12 — Masked Language Modeling (Actividad 6)
Enmascaramos un aminoácido y le pedimos al modelo que prediga cuál es.
Como ESM-2 es encoder bidireccional, puede ver AMBOS lados de la máscara.

In [ ]:
# ============================================================
# CELDA 12: MASKED LANGUAGE MODELING CON TOP-K
# ============================================================

print("MASKED LANGUAGE MODELING")
print("="*60)

# Cargamos el modelo MLM (tiene cabeza de clasificación adicional)
print("Cargando EsmForMaskedLM...")
mlm_model = EsmForMaskedLM.from_pretrained(MODEL_NAME)
mlm_model = mlm_model.to(device)
mlm_model.eval()

# Enmascaramos la G (glicina) en la posición 76 de la Ubiquitina
seq_original = sequences["Original"]
mask_token   = tokenizer.mask_token  # '<mask>'

# Posición 76 es el índice 75 (Python empieza en 0)
pos_G = 75
masked_seq = seq_original[:pos_G] + mask_token + seq_original[pos_G+1:]

print(f"  Secuencia original:  {seq_original}")
print(f"  Secuencia mascarada: {masked_seq}")
print(f"  Token enmascarado:   posición 76 (aminoácido '{seq_original[pos_G]}')")

# Tokenizamos la secuencia enmascarada
masked_inputs = tokenizer(masked_seq, return_tensors="pt")
masked_inputs = {k: v.to(device) for k, v in masked_inputs.items()}

# Encontramos la posición del token máscara en el tensor de IDs
mask_token_id  = tokenizer.mask_token_id
mask_positions = (masked_inputs["input_ids"] == mask_token_id).nonzero(as_tuple=True)[1]
mask_pos_in_tensor = mask_positions[0].item()

print(f"  Posición del <mask> en el tensor: {mask_pos_in_tensor}")

# Inferencia con el modelo MLM
with torch.no_grad():
    mlm_outputs = mlm_model(**masked_inputs)

# Extraemos los logits para la posición enmascarada
logits_mask = mlm_outputs.logits[0, mask_pos_in_tensor, :]  # (vocab_size,)

# Top-26 predicciones
K = 26
topk = torch.topk(logits_mask, k=K)

print(f"\n   TOP-{K} PREDICCIONES para la posición 76:")
print(f"  {'Rango':<5} {'Token':<8} {'Score (logit)':<15} {'Aminoácido'}")
print(f"  {'-'*50}")
for rank, (score, token_id) in enumerate(zip(topk.values, topk.indices), 1):
    token = tokenizer.convert_ids_to_tokens([token_id.item()])[0]
    es_correcto = "✅ (correcto)" if token == seq_original[pos_G] else ""
    print(f"  {rank:<6} {token:<8} {float(score):<15.4f} {es_correcto}")

In [ ]:
# ============================================================
# CELDA 12B: GRÁFICA DE TOP-K PREDICCIONES
# ============================================================

tokens_pred = [tokenizer.convert_ids_to_tokens([topk.indices[i].item()])[0] for i in range(K)]
scores_pred = [float(topk.values[i]) for i in range(K)]
colors_bar  = ["gold" if t == seq_original[pos_G] else "steelblue" for t in tokens_pred]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(tokens_pred[::-1], scores_pred[::-1], color=colors_bar[::-1], edgecolor='white')
ax.set_xlabel("Logit score", fontsize=11)
ax.set_title(f"Top-{K} predicciones para posición 76 (original: '{seq_original[pos_G]}')",
             fontsize=11)

# Leyenda
patch_correct = mpatches.Patch(color='gold', label=f"Token correcto: '{seq_original[pos_G]}'")
patch_other   = mpatches.Patch(color='steelblue', label='Otros candidatos')
ax.legend(handles=[patch_correct, patch_other], fontsize=9)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig("mlm_topk.png", dpi=150, bbox_inches='tight')
plt.show()
print("Figura guardada como mlm_topk.png")

---
## Celda 13 — Diferencia de hidden states entre capas (visualización evolutiva)

In [ ]:
# ============================================================
# CELDA 13: EVOLUCIÓN DEL EMBEDDING DEL TOKEN Y A TRAVÉS DE CAPAS
# ============================================================
# Mostramos cómo el vector del aminoácido Y cambia capa a capa,
# lo que ilustra que las representaciones se 'contextualizan' progresivamente.

print(" EVOLUCIÓN DEL HIDDEN STATE DEL TOKEN 'G' (pos.5) A TRAVÉS DE CAPAS")
print("="*60)

outputs_orig = outputs_dict["Original"]
# hidden_states: tupla de (num_layers+1) tensores, cada uno (1, seq_len, hidden_size)
# [0] = embeddings iniciales, [1..6] = salidas de cada capa transformer

token_pos = pos_G + 1  # +1 por el token CLS al inicio

norms = []
for layer_i, hs in enumerate(outputs_orig.hidden_states):
    vec = hs[0, token_pos, :].cpu().numpy()
    norms.append(np.linalg.norm(vec))
    print(f"  Capa {layer_i:2d}: norma del vector G = {norms[-1]:.4f}")

# También comparamos norma de Y en secuencia original vs mutada
print("\n  Comparación en la ÚLTIMA CAPA (capa 6):")
for nombre, out in outputs_dict.items():
    # En la mutada, la posición 5 ahora es F, pero comparamos la misma posición
    vec = out.hidden_states[-1][0, token_pos, :].cpu().numpy()
    print(f"    {nombre}: norma en pos {pos_G} = {np.linalg.norm(vec):.4f}")

# Gráfica de evolución de normas
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(len(norms)), norms, marker='o', color='steelblue', linewidth=2)
ax.set_xlabel("Capa (0 = embeddings iniciales, 1-6 = capas transformer)", fontsize=10)
ax.set_ylabel("Norma L2 del vector", fontsize=10)
ax.set_title(f"Evolución del hidden state del token 'G' (posición {pos_G}) a través de capas",
             fontsize=11)
ax.set_xticks(range(len(norms)))
ax.set_xticklabels([f"Capa {i}" for i in range(len(norms))], rotation=30)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("hidden_state_evolution.png", dpi=150, bbox_inches='tight')
plt.show()
print("Figura guardada como hidden_state_evolution.png")

---
## Celda 14 — Resumen final y usos reales (Actividad 7)

In [ ]:
# ============================================================
# CELDA 14: RESUMEN EJECUTIVO Y CONCLUSIONES
# ============================================================

print("="*70)
print("RESUMEN EJECUTIVO DEL NOTEBOOK")
print("="*70)

print("""
MODELO USADO:
  facebook/esm2_t6_8M_UR50D
  - 6 capas transformer encoder
  - 20 cabezas de atención por capa
  - Hidden size: 320 dimensiones
  - ~8 millones de parámetros

SECUENCIAS ANALIZADAS:
  Original:  MKTAYIAKQRQISFVKSHFSRQDILD (26 aa)
  Mutada:    MKTAFIAKQRQISFVKSHFSRQDILD (26 aa) — Y→F en posición 5
  Alterada:  DLIDQRSFHSSKVFSIQRQKAIYATKM (27 aa) — inversión parcial

EVIDENCIAS OBTENIDAS:
  Tokenización: IDs, tokens especiales, shapes
  last_hidden_state extraído para las 3 secuencias
  hidden_states de todas las capas (0 a 6)
  Matrices de atención (capa temprana, profunda, dos cabezas, orig vs mutada)
  Embeddings globales por proteína (promedio de residuos)
  Similitud coseno entre secuencias
  PCA en 2D de embeddings por residuo
  Masked Language Modeling top-5
  Evolución del hidden state a través de capas
  Inspección de módulos internos (Q, K, V, embeddings, MLM head)

USOS REALES DE ESM-2:
  1. Comparación de proteínas por similitud coseno entre embeddings globales
  2. Análisis de mutaciones: detectar cambios en representaciones ante mutaciones puntuales
  3. Base para predicción estructural (ESMFold usa ESM-2 internamente)
  4. Clasificación funcional: los embeddings alimentan clasificadores downstream
  5. Búsqueda semántica de proteínas en bases de datos vectoriales
  6. Priorización de variantes para ingeniería de proteínas

LIMITACIONES IMPORTANTES:
  - Los mapas de atención NO prueban causalidad biológica
  - ESM-2 NO reemplaza validación experimental en laboratorio
  - Los modelos grandes requieren GPU; el modelo de 8M es solo para docencia
  - Sesgos hacia secuencias bien representadas en UniRef50 (datos de entrenamiento)
  - Las representaciones son estadísticas, no mecanismos biológicos
""")

print("="*70)
print("Notebook completo. Todas las actividades cubiertas.")
print("="*70)

---
# AJUSTANDO LAS APLICACIONES
---
### MINI BUSCADOR DE PROTEÍNAS
Se tiene una consulta y el sistema se devuelve cuáles otras proteínas son más similares a ellas, ordenadas por similitud de Coseno. Como Google para proteínas.

In [ ]:
protein_database = {
    "Ubiquitina":      "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG",
    "Ubiquitina_G76A": "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGA",
    "SUMO1":           "MSDQEAKPSTEDLGDKKEGEYIKLKVIGQDSSEIHFKVKMTTHLKKLKESYCQRQGVPMNSLRFLLFEGQRIADNHTPKELGMEEEDVIEVYQEQTGGHSTV",
    "H3_Histona":      "ARTKQTARKSTGGKAPRKQLATKAARKSAPATGGVKKPHRYRPGTVALREIRRYQKSTELLIRKLPFQRLVREIAQDFKTDLRFQSSAVMALQEACEAYLVGLFEDTNLCAIHAKRVTIMPKDIQLARRIRGERA",
    "CaM_Calmodulina": "MADQLTEEQIAEFKEAFSLFDKDGDGTITTKELGTVMRSLGQNPTEAELQDMINEVDADGNGTIDFPEFLTMMARKMKDTDSEEEIREAFRVFDKDGNGYISAAELRHVMTNLGEKLTDEEVDEMIREADIDGDGQVNYEEFVQMMTAK",
    "Lisozima":        "KVFGRCELAAAAMKRHGLDNYRGYSLGNWVCAAKFESNFNTQATNRNTDGSTDYGILQINSRWWCNDGRTPGSRNLCNIPCSALLSSDITASVNCAKKIVSDGNGMNAWVAWRNRCKGTDVQAWIRGCRL",
    "Mioglobina":      "MGLSDGEWQLVLNVWGKVEADIPGHGQEVLIRLFKGHPETLEKFDKFKHLKSEDEMKASEDLKKHGATVLTALGGILKKKGHHEAEIKPLAQSHATKHKIPVKYLEFISECIIQVLQSKHPGDFGADAQGAMNKALELFRKDMASNYKELGFQG",
    "Insulina_B":      "FVNQHLCGSHLVEALYLVCGERGFFYTPKT",
    "Tirosina_quinasa": "MKKFFDSRREQGGSGLGSGSSGGGGSTSGLGSGYIGRVFGIGRQQVTVDEVLAEGGFAIVFLVRTSNGMKCALKRMFVNNTEGVREALALGSELPQIPAELQIMAHQLAREGYVHRDLAARNVLVKTPQHVKITDFGLAKLLGAEEKEYHAEGGKVPIKWMALESILHRIYTHQSDVWSYGVVLWEIFSLGGSPYPGVQITEECWDFLRGKNNPVLKDVVMGDTFNNKOSLNQVVQDKFPVNQEGIRQLNMEREQNLSSARQLEKVDERLRRNILLEKNPPDIEQALRELEKDPQQAEALSLDQAQKVAGLGAQHFSECDFQLEQLQKQMGSFK",
    "Proteina_C":      "MRVGGKPGSGKTTYVKKHLDGSVTRSAPALQNLKAQIAAELHAEGKIPFRVKSGDRFSESGYKGWYDFDLESPKQMTIQEVREYFHPVHVQFDDYPLNQLREQLYRAVKKQGKPIEVWKYNPDTKTDGKMIISTLGRPKIEGKFAIINGKNSFDKSNVQASAAEISKLINKISQPEWLTTFRPFKKSPMLIEQELENTLQRVQHESILQDKVLNLTNEHVLMIQAYQDQFHTELSTQETLKKQLHEAFGMKSFQDTINQVFNQLNKVEIKFNNQLAFMNHPQIPQQFMSQLNRMQEALQRQTRSQLQEAQDQLEAIEMQHQSTQEQINVLFQQQHQQQLNQLAELQARLQPQQAQMIQQQSMQMKQSEIESLKQSQEQLQKQREQQRDLEQQLNQQKNMIQKMQQQLEQLLQQQQQQSHQQQLKQLEQLRQQSQMQLRQMQAEQQQQLQTMQQLQQKLQEQMKQLQAQIQQLQMQMQLQQVQMQLRNHQMSIMQQSLQQKGQE"
}

In [ ]:
# ============================================================
# CELDA 15: BÚSQUEDA SEMÁNTICA INTERACTIVA CON UNIPROT
# ============================================================

import requests
import time
import ipywidgets as widgets
from IPython.display import display, clear_output

# ---- Función: obtener proteína consulta de UniProt por nombre ----
def obtener_proteina_consulta(nombre):
    """
    Busca UNA proteína específica en UniProt por nombre
    y devuelve su secuencia.
    """
    url = "https://rest.uniprot.org/uniprotkb/search"
    params = {
        "query": f"{nombre} AND reviewed:true",
        "format": "json",
        "size": 1,
        "fields": "accession,protein_name,sequence,organism_name"
    }
    response = requests.get(url, params=params)
    if response.status_code != 200:
        return None, None, None
    data = response.json()
    resultados = data.get("results", [])
    if not resultados:
        return None, None, None
    entry = resultados[0]
    try:
        nombre_oficial = entry["proteinDescription"]["recommendedName"]["fullName"]["value"]
        secuencia      = entry["sequence"]["value"]
        organismo      = entry["organism"]["scientificName"]
        accession      = entry["primaryAccession"]
        return f"{nombre_oficial} [{accession}]", secuencia, organismo
    except KeyError:
        return None, None, None

# ---- Función: buscar proteínas similares en UniProt ----
def buscar_proteinas_relacionadas(query, max_resultados=15):
    url = "https://rest.uniprot.org/uniprotkb/search"
    params = {
        "query": f"{query} AND reviewed:true AND length:[50 TO 250]",
        "format": "json",
        "size": max_resultados,
        "fields": "accession,protein_name,sequence,organism_name"
    }
    response = requests.get(url, params=params)
    if response.status_code != 200:
        return []
    data = response.json()
    resultados = []
    for entry in data.get("results", []):
        try:
            nombre   = entry["proteinDescription"]["recommendedName"]["fullName"]["value"]
            seq      = entry["sequence"]["value"]
            acc      = entry["primaryAccession"]
            org      = entry["organism"]["scientificName"]
            resultados.append((f"{nombre[:28]} [{acc}]", seq, org))
        except KeyError:
            continue
    return resultados

# ---- Función: calcular embedding global ----
def get_embedding(seq):
    inputs = tokenizer(seq, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model(**inputs, output_hidden_states=True)
    hidden = out.hidden_states[-1][0]
    return hidden[1:hidden.shape[0]-1].mean(dim=0).cpu().numpy()

# ---- Función principal del botón ----
def ejecutar_busqueda(boton):
    with output:
        clear_output()

        nombre_consulta = campo_busqueda.value.strip()
        if not nombre_consulta:
            print("⚠️  Escribe el nombre de una proteína para buscar.")
            return

        # Paso 1: obtener la proteína consulta
        print(f"⏳ Buscando '{nombre_consulta}' en UniProt...")
        nombre_oficial, secuencia_consulta, organismo = obtener_proteina_consulta(nombre_consulta)

        if not secuencia_consulta:
            print(f"❌ No se encontró '{nombre_consulta}' en UniProt.")
            print("   Intenta con otro nombre, por ejemplo: 'ubiquitin human', 'insulin', 'hemoglobin'")
            return

        print(f"\n✅ Proteína encontrada:")
        print(f"   Nombre:    {nombre_oficial}")
        print(f"   Organismo: {organismo}")
        print(f"   Secuencia: {secuencia_consulta[:50]}... ({len(secuencia_consulta)} aa)")

        # Paso 2: buscar proteínas relacionadas
        print(f"\n⏳ Buscando proteínas relacionadas en UniProt...")
        proteinas = buscar_proteinas_relacionadas(nombre_consulta, max_resultados=15)

        if not proteinas:
            print("❌ No se encontraron proteínas relacionadas.")
            return

        print(f"✅ {len(proteinas)} proteínas obtenidas\n")

        # Paso 3: calcular embeddings
        print("⏳ Calculando embeddings con ESM-2...")
        db_embeddings = {}
        for nombre, seq, org in proteinas:
            try:
                db_embeddings[nombre] = get_embedding(seq)
                print(f"  ✅ {nombre[:45]} ({len(seq)} aa)")
            except Exception as e:
                print(f"  ⚠️  Error en {nombre}: {e}")
            time.sleep(0.1)

        # Paso 4: calcular similitudes
        print("\n📊 Calculando similitudes coseno...")
        query_vec = get_embedding(secuencia_consulta).reshape(1, -1)

        resultados = []
        for nombre, vec in db_embeddings.items():
            sim = cosine_similarity(query_vec, vec.reshape(1, -1))[0][0]
            resultados.append((nombre, sim))
        resultados.sort(key=lambda x: x[1], reverse=True)

        # Paso 5: tabla de resultados
        print(f"\n  {'Rango':<6} {'Similitud':<12} {'Proteína'}")
        print(f"  {'-'*70}")
        for i, (nombre, sim) in enumerate(resultados, 1):
            print(f"  {i:<6} {sim:.4f}      {nombre}")

        # Paso 6: gráfica
        nombres_res = [r[0][:35] for r in resultados[:10]]
        sims_res    = [r[1] for r in resultados[:10]]

        fig, ax = plt.subplots(figsize=(11, 6))
        bars = ax.barh(nombres_res[::-1], sims_res[::-1],
                       color="steelblue", edgecolor="white")
        ax.set_xlabel("Similitud coseno", fontsize=11)
        ax.set_title(f"Top 10 proteínas más similares a '{nombre_oficial[:40]}'",
                     fontsize=12)
        ax.set_xlim(0.5, 1.0)
        ax.axvline(x=0.95, color='red', linestyle='--',
                   alpha=0.6, label='Alta similitud (0.95)')
        ax.axvline(x=0.75, color='orange', linestyle='--',
                   alpha=0.6, label='Similitud media (0.75)')
        for bar, sim in zip(bars, sims_res[::-1]):
            ax.text(bar.get_width() + 0.002,
                    bar.get_y() + bar.get_height()/2,
                    f'{sim:.4f}', va='center', fontsize=8)
        ax.legend(fontsize=9)
        ax.grid(axis='x', alpha=0.3)
        plt.tight_layout()
        plt.savefig("uniprot_semantic_search.png", dpi=150, bbox_inches='tight')
        plt.show()
        print("💾 Figura guardada como uniprot_semantic_search.png")

# ---- Interfaz — solo un campo ----
campo_busqueda = widgets.Text(
    value='ubiquitin human',
    description='🔎 Proteína:',
    placeholder='Ej: ubiquitin human, insulin, hemoglobin',
    layout=widgets.Layout(width='500px')
)

boton = widgets.Button(
    description='🔍 Buscar proteínas similares',
    button_style='primary',
    layout=widgets.Layout(width='300px', height='40px')
)

output = widgets.Output()
boton.on_click(ejecutar_busqueda)

print("="*60)
print("🔬 BUSCADOR SEMÁNTICO DE PROTEÍNAS — UniProt + ESM-2")
print("="*60)
print("Escribe el nombre de una proteína y presiona el botón.")
print("El sistema encontrará automáticamente su secuencia")
print("y buscará las proteínas más similares en UniProt.\n")
display(campo_busqueda, boton, output)

In [ ]:
# ============================================================
# CELDA 16: PRIORIZACIÓN DE VARIANTES CON WIDGET INTERACTIVO
# ============================================================

import random
import requests
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd

AMINOACIDOS = list("ACDEFGHIKLMNPQRSTVWY")
historial   = []

# ---- Función: obtener secuencia de UniProt ----
def obtener_secuencia_uniprot(nombre):
    url    = "https://rest.uniprot.org/uniprotkb/search"
    params = {
        "query":  f"{nombre} AND reviewed:true",
        "format": "json",
        "size":   1,
        "fields": "accession,protein_name,sequence,organism_name"
    }
    response = requests.get(url, params=params)
    if response.status_code != 200:
        return None, None, None
    data      = response.json()
    resultados = data.get("results", [])
    if not resultados:
        return None, None, None
    entry = resultados[0]
    try:
        nombre_oficial = entry["proteinDescription"]["recommendedName"]["fullName"]["value"]
        secuencia      = entry["sequence"]["value"]
        accession      = entry["primaryAccession"]
        return nombre_oficial, secuencia, accession
    except KeyError:
        return None, None, None

# ---- Función: calcular embedding ----
def get_embedding(seq):
    inputs = tokenizer(seq, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model(**inputs, output_hidden_states=True)
    hidden = out.hidden_states[-1][0]
    return hidden[1:hidden.shape[0]-1].mean(dim=0).cpu().numpy()

# ---- Función: generar mutación aleatoria ----
def generar_mutacion(seq):
    pos       = random.randint(0, len(seq) - 1)
    aa_orig   = seq[pos]
    aa_nuevo  = random.choice([aa for aa in AMINOACIDOS if aa != aa_orig])
    seq_mut   = seq[:pos] + aa_nuevo + seq[pos+1:]
    return pos, aa_orig, aa_nuevo, seq_mut

# ---- Estado compartido entre botones ----
estado = {"secuencia": None, "nombre": None, "embedding": None}

# ---- Botón: cargar proteína ----
def cargar_proteina(boton):
    with output:
        clear_output()
        historial.clear()

        nombre_input = campo_proteina.value.strip()
        if not nombre_input:
            print("⚠️  Escribe el nombre de una proteína primero.")
            return

        print(f"⏳ Buscando '{nombre_input}' en UniProt...")
        nombre_oficial, secuencia, accession = obtener_secuencia_uniprot(nombre_input)

        if not secuencia:
            print(f"❌ No se encontró '{nombre_input}' en UniProt.")
            print("   Intenta con: 'ubiquitin human', 'insulin human', 'hemoglobin human'")
            return

        print(f"\n✅ Proteína cargada:")
        print(f"   Nombre:    {nombre_oficial} [{accession}]")
        print(f"   Secuencia: {secuencia[:50]}... ({len(secuencia)} aa)")
        print(f"\n⏳ Calculando embedding base...")

        estado["nombre"]    = f"{nombre_oficial} [{accession}]"
        estado["secuencia"] = secuencia
        estado["embedding"] = get_embedding(secuencia).reshape(1, -1)

        print(f"✅ Listo. Ahora puedes generar mutaciones.")

# ---- Botón: generar mutaciones ----
def ejecutar_mutaciones(boton):
    with output:
        clear_output()

        if estado["secuencia"] is None:
            print("⚠️  Primero carga una proteína usando el botón 'Cargar proteína'.")
            return

        n = slider_cantidad.value
        print(f"🧬 Generando {n} mutaciones aleatorias sobre: {estado['nombre']}")
        print("="*60)

        for _ in range(n):
            pos, aa_orig, aa_nuevo, seq_mut = generar_mutacion(estado["secuencia"])
            emb_mut   = get_embedding(seq_mut).reshape(1, -1)
            sim       = cosine_similarity(estado["embedding"], emb_mut)[0][0]
            disrupcion = 1 - sim

            historial.append({
                "Mutación":   f"{aa_orig}{pos+1}{aa_nuevo}",
                "Posición":   pos + 1,
                "Original":   aa_orig,
                "Nueva":      aa_nuevo,
                "Similitud":  round(float(sim), 4),
                "Disrupción": round(float(disrupcion), 4)
            })
            print(f"  ✅ {aa_orig}{pos+1}{aa_nuevo} → similitud: {sim:.4f}  disrupción: {disrupcion:.4f}")

        # Tabla acumulativa ordenada
        historial_ordenado = sorted(historial, key=lambda x: x["Disrupción"], reverse=True)

        print(f"\n📋 TABLA ACUMULATIVA ({len(historial)} mutaciones probadas)")
        print("="*60)
        df = pd.DataFrame(historial_ordenado)
        print(df.to_string(index=False))

        # Gráfica top 15
        top     = historial_ordenado[:15]
        nombres = [r["Mutación"] for r in top]
        disrups = [r["Disrupción"] for r in top]
        colores = ["crimson"    if d > 0.05 else
                   "orange"     if d > 0.02 else
                   "steelblue"  for d in disrups]

        fig, ax = plt.subplots(figsize=(10, 6))
        bars = ax.barh(nombres[::-1], disrups[::-1],
                       color=colores[::-1], edgecolor="white")
        ax.set_xlabel("Disrupción (1 - similitud coseno)", fontsize=11)
        ax.set_title(f"Top mutaciones más disruptivas — {estado['nombre']}\n"
                     f"{len(historial)} variantes probadas", fontsize=11)
        ax.axvline(x=0.05, color='crimson', linestyle='--',
                   alpha=0.6, label='Alta disrupción (>0.05)')
        ax.axvline(x=0.02, color='orange',  linestyle='--',
                   alpha=0.6, label='Disrupción media (>0.02)')
        for bar, d in zip(bars, disrups[::-1]):
            ax.text(bar.get_width() + 0.001,
                    bar.get_y() + bar.get_height()/2,
                    f'{d:.4f}', va='center', fontsize=8)
        ax.legend(fontsize=9)
        ax.grid(axis='x', alpha=0.3)
        plt.tight_layout()
        plt.savefig("variantes_disruptivas.png", dpi=150, bbox_inches='tight')
        plt.show()
        print("💾 Figura guardada como variantes_disruptivas.png")

# ---- Botón: limpiar historial ----
def limpiar_historial(boton):
    with output:
        historial.clear()
        clear_output()
        print("🗑️  Historial limpiado. Puedes generar nuevas mutaciones.")

# ---- Interfaz ----
campo_proteina = widgets.Text(
    value='ubiquitin human',
    description='🧬 Proteína:',
    placeholder='Ej: ubiquitin human, insulin human, hemoglobin',
    layout=widgets.Layout(width='500px')
)

boton_cargar = widgets.Button(
    description='⬇️ Cargar proteína',
    button_style='info',
    layout=widgets.Layout(width='200px', height='40px')
)

slider_cantidad = widgets.IntSlider(
    value=5,
    min=1,
    max=20,
    step=1,
    description='# Mutaciones:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

boton_generar = widgets.Button(
    description='🧬 Generar mutaciones',
    button_style='primary',
    layout=widgets.Layout(width='220px', height='40px')
)

boton_limpiar = widgets.Button(
    description='🗑️ Limpiar historial',
    button_style='danger',
    layout=widgets.Layout(width='200px', height='40px')
)

output = widgets.Output()

boton_cargar.on_click(cargar_proteina)
boton_generar.on_click(ejecutar_mutaciones)
boton_limpiar.on_click(limpiar_historial)

print("="*60)
print("🧬 PRIORIZACIÓN DE VARIANTES")
print("="*60)
print("1. Escribe el nombre de una proteína y presiona 'Cargar'")
print("2. Elige cuántas mutaciones generar con el slider")
print("3. Presiona 'Generar mutaciones' para ver los resultados\n")

display(
    campo_proteina,
    boton_cargar,
    slider_cantidad,
    widgets.HBox([boton_generar, boton_limpiar]),
    output
)

In [ ]:
# ============================================================
# CELDA 17: CLASIFICACIÓN FUNCIONAL CON UNIPROT + ESM-2
# ============================================================

import requests
import time
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder

# ---- Categorías funcionales y términos de búsqueda en UniProt ----
CATEGORIAS = {
    "hormona":    "hormone AND reviewed:true AND length:[20 TO 200]",
    "enzima":     "enzyme AND reviewed:true AND length:[50 TO 200]",
    "anticuerpo": "immunoglobulin AND reviewed:true AND length:[50 TO 200]",
    "transporte": "transport protein AND reviewed:true AND length:[50 TO 200]",
    "estructural": "structural protein AND reviewed:true AND length:[50 TO 200]"
}

N_POR_CATEGORIA = 8  # proteínas de entrenamiento por categoría

# ---- Estado compartido ----
estado = {
    "clasificador": None,
    "encoder":      None,
    "entrenado":    False
}

# ---- Función: obtener proteínas de UniProt por categoría ----
def obtener_proteinas_categoria(query, n=8):
    url    = "https://rest.uniprot.org/uniprotkb/search"
    params = {
        "query":  query,
        "format": "json",
        "size":   n,
        "fields": "accession,protein_name,sequence"
    }
    response = requests.get(url, params=params)
    if response.status_code != 200:
        return []
    data = response.json()
    resultados = []
    for entry in data.get("results", []):
        try:
            nombre = entry["proteinDescription"]["recommendedName"]["fullName"]["value"]
            seq    = entry["sequence"]["value"]
            acc    = entry["primaryAccession"]
            resultados.append((f"{nombre[:25]} [{acc}]", seq))
        except KeyError:
            continue
    return resultados

# ---- Función: calcular embedding ----
def get_embedding(seq):
    inputs = tokenizer(seq, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model(**inputs, output_hidden_states=True)
    hidden = out.hidden_states[-1][0]
    return hidden[1:hidden.shape[0]-1].mean(dim=0).cpu().numpy()

# ---- Función: obtener secuencia de UniProt por nombre ----
def obtener_secuencia(nombre):
    url    = "https://rest.uniprot.org/uniprotkb/search"
    params = {
        "query":  f"{nombre} AND reviewed:true",
        "format": "json",
        "size":   1,
        "fields": "accession,protein_name,sequence"
    }
    response = requests.get(url, params=params)
    if response.status_code != 200:
        return None, None
    data = response.json()
    resultados = data.get("results", [])
    if not resultados:
        return None, None
    entry = resultados[0]
    try:
        nombre_oficial = entry["proteinDescription"]["recommendedName"]["fullName"]["value"]
        seq            = entry["sequence"]["value"]
        return nombre_oficial, seq
    except KeyError:
        return None, None

# ---- Botón: entrenar clasificador ----
def entrenar_clasificador(boton):
    with output:
        clear_output()

        print("🏋️  ENTRENANDO CLASIFICADOR")
        print("="*60)
        print(f"Descargando {N_POR_CATEGORIA} proteínas por cada una de las {len(CATEGORIAS)} categorías...\n")

        X, y = [], []

        for categoria, query in CATEGORIAS.items():
            print(f"⏳ Descargando proteínas de categoría: '{categoria}'...")
            proteinas = obtener_proteinas_categoria(query, N_POR_CATEGORIA)

            if not proteinas:
                print(f"  ⚠️  No se encontraron proteínas para '{categoria}'")
                continue

            for nombre, seq in proteinas:
                try:
                    emb = get_embedding(seq)
                    X.append(emb)
                    y.append(categoria)
                    print(f"  ✅ [{categoria}] {nombre[:45]} ({len(seq)} aa)")
                except Exception as e:
                    print(f"  ⚠️  Error en {nombre}: {e}")
                time.sleep(0.1)

        if len(X) < 5:
            print("\n❌ No hay suficientes proteínas para entrenar. Intenta de nuevo.")
            return

        # Entrenar clasificador KNN
        X_array = np.array(X)
        encoder = LabelEncoder()
        y_encoded = encoder.fit_transform(y)

        clf = KNeighborsClassifier(n_neighbors=3, metric='cosine')
        clf.fit(X_array, y_encoded)

        estado["clasificador"] = clf
        estado["encoder"]      = encoder
        estado["entrenado"]    = True

        print(f"\n✅ Clasificador entrenado con {len(X)} proteínas")
        print(f"   Categorías: {list(encoder.classes_)}")
        print(f"\n🎯 Listo. Ahora puedes predecir la categoría de cualquier proteína.")

# ---- Botón: predecir ----
def predecir_proteina(boton):
    with output:
        clear_output()

        if not estado["entrenado"]:
            print("⚠️  Primero entrena el clasificador presionando 'Entrenar'.")
            return

        nombre_input = campo_prediccion.value.strip()
        if not nombre_input:
            print("⚠️  Escribe el nombre de una proteína para predecir.")
            return

        print(f"🔎 Buscando '{nombre_input}' en UniProt...")
        nombre_oficial, seq = obtener_secuencia(nombre_input)

        if not seq:
            print(f"❌ No se encontró '{nombre_input}' en UniProt.")
            return

        print(f"✅ Proteína encontrada: {nombre_oficial} ({len(seq)} aa)")
        print(f"⏳ Calculando embedding...")

        emb = get_embedding(seq).reshape(1, -1)

        # Predecir categoría
        pred_encoded   = estado["clasificador"].predict(emb)
        pred_categoria = estado["encoder"].inverse_transform(pred_encoded)[0]

        # Probabilidades de cada categoría
        distancias, indices = estado["clasificador"].kneighbors(emb)
        vecinos_labels = [estado["encoder"].classes_[
            estado["clasificador"]._y[i]] for i in indices[0]]

        print(f"\n{'='*60}")
        print(f"🏷️  RESULTADO DE CLASIFICACIÓN")
        print(f"{'='*60}")
        print(f"   Proteína:  {nombre_oficial}")
        print(f"   Categoría predicha: 👉 {pred_categoria.upper()}")
        print(f"\n   Vecinos más cercanos:")
        for i, (vecino, dist) in enumerate(zip(vecinos_labels, distancias[0]), 1):
            print(f"   {i}. Categoría: {vecino:<15} distancia: {dist:.4f}")

        # Gráfica de confianza por categoría
        conteo = {cat: vecinos_labels.count(cat) for cat in estado["encoder"].classes_}
        total  = sum(conteo.values())
        confianzas = {cat: v/total for cat, v in conteo.items()}

        categorias_graf = list(confianzas.keys())
        valores_graf    = list(confianzas.values())
        colores_graf    = ["gold" if c == pred_categoria else "steelblue"
                           for c in categorias_graf]

        fig, ax = plt.subplots(figsize=(8, 4))
        bars = ax.barh(categorias_graf, valores_graf,
                       color=colores_graf, edgecolor="white")
        ax.set_xlabel("Confianza del clasificador", fontsize=11)
        ax.set_title(f"Clasificación funcional — '{nombre_oficial[:40]}'", fontsize=11)
        ax.set_xlim(0, 1.1)
        for bar, val in zip(bars, valores_graf):
            ax.text(bar.get_width() + 0.02,
                    bar.get_y() + bar.get_height()/2,
                    f'{val*100:.0f}%', va='center', fontsize=10)
        ax.grid(axis='x', alpha=0.3)
        plt.tight_layout()
        plt.savefig("clasificacion_funcional.png", dpi=150, bbox_inches='tight')
        plt.show()
        print("💾 Figura guardada como clasificacion_funcional.png")

# ---- Interfaz ----
boton_entrenar = widgets.Button(
    description='🏋️ Entrenar clasificador',
    button_style='warning',
    layout=widgets.Layout(width='250px', height='40px')
)

campo_prediccion = widgets.Text(
    value='ubiquitin human',
    description='🧬 Proteína:',
    placeholder='Ej: insulin human, collagen, myoglobin',
    layout=widgets.Layout(width='500px')
)

boton_predecir = widgets.Button(
    description='🎯 Predecir categoría',
    button_style='primary',
    layout=widgets.Layout(width='220px', height='40px')
)

output = widgets.Output()

boton_entrenar.on_click(entrenar_clasificador)
boton_predecir.on_click(predecir_proteina)

print("="*60)
print("🏷️  CLASIFICADOR FUNCIONAL DE PROTEÍNAS — UniProt + ESM-2")
print("="*60)
print("Paso 1: Presiona 'Entrenar' para construir el clasificador")
print("        (descarga proteínas reales de UniProt — tarda ~2 min)")
print("Paso 2: Escribe cualquier proteína y predice su categoría\n")

display(
    boton_entrenar,
    widgets.Label(value="─"*50),
    campo_prediccion,
    boton_predecir,
    output
)